# Eksploracja: chunker (v3)

Notebook do ręcznej weryfikacji chunkera po dostosowaniu do ekstrakcji v2:
- media = `{table, infographic}` (osobne chunki, caption nad nimi = prefiks treści),
- skip = `{picture}` (pomijane; nie przerywają tekstu),
- `footnote`, `identifier`, `subsection-header`, osierocony `caption` — akumulowane w chunku tekstowym,
- `section-header` twardo tnie chunk tekstowy,
- overlap wyłącznie w trybie znakowym, w obrębie sekcji.

Przepływ:
1. Wybierz zakres — pojedynczy rozdział (`"I"` … `"X"`) lub `"ALL"`.
2. Ustaw parametry chunkingu.
3. Obejrzyj rozkład typów bloków / chunków.
4. Sanity-check: pojawia się `infographic`, nie pojawia się `picture`, `footnote`/`identifier` wpletone w `text`.
5. Podgląd mediów (table/infographic) i chunków tekstowych.
6. Weryfikacja overlapu.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from collections import Counter
from pathlib import Path
from statistics import mean, median

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.app.document.models import (
    DocumentMetadata,
    ExtractedChapter,
    ExtractedDocument,
)
from backend.app.document.vision_chunker import chunk_document

CACHE_DIR = ROOT / "data" / "extraction_v2"
CHAPTER_ORDER = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]
available = [cid for cid in CHAPTER_ORDER if (CACHE_DIR / f"{cid}.json").exists()]
print("Dostępne rozdziały:", available)

Dostępne rozdziały: ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X']


## 2. Wybór zakresu

Ustaw `SELECTION` na ID rozdziału (`"I"`…`"X"`) albo `"ALL"` dla całego dokumentu, potem re-run komórki.

In [2]:
SELECTION = "ALL"  # np. "I", "V", "ALL"


def load_selection(sel: str) -> ExtractedDocument:
    ids = available if sel == "ALL" else [sel]
    chapters = [ExtractedChapter.load_json(CACHE_DIR / f"{cid}.json") for cid in ids]
    total_pages = sum(ch.page_end - ch.page_start + 1 for ch in chapters)
    title = "BGK 2024 v2 (ALL)" if sel == "ALL" else f"BGK 2024 v2 — rozdz. {sel}"
    return ExtractedDocument(
        metadata=DocumentMetadata(
            source_file="vision_cache_v2",
            total_pages=total_pages,
            extraction_date="",
        ),
        title=title,
        chapters=chapters,
    )


doc = load_selection(SELECTION)
block_types = Counter(b.element_type for b in doc.get_all_blocks())
print(f"Zakres:       {SELECTION}")
print(f"Rozdziały:    {len(doc.chapters)} ({[ch.chapter_id for ch in doc.chapters]})")
print(f"Stron:        {len(doc.get_all_pages())}")
print(f"Bloków:       {sum(block_types.values())}")
print("Typy bloków:")
for t, n in block_types.most_common():
    print(f"  {t:<20} {n}")

Zakres:       ALL
Rozdziały:    10 (['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X'])
Stron:        182
Bloków:       2242
Typy bloków:
  text                 1071
  subsection-header    362
  list                 209
  identifier           133
  table                108
  picture              105
  caption              94
  footnote             59
  section-header       57
  infographic          44


## 3. Parametry chunkingu

`overlap_mode` został wycofany — overlap zawsze w trybie znakowym.

In [3]:
MAX_CHARS = 1500
OVERLAP_CHARS = 200

chunks = chunk_document(
    doc,
    max_chars=MAX_CHARS,
    overlap_chars=OVERLAP_CHARS,
)
chunk_types = Counter(c["element_type"] for c in chunks)
print(f"Liczba chunków: {len(chunks)}")
for t, n in chunk_types.most_common():
    print(f"  {t:<12} {n}")

assert "picture" not in chunk_types, "picture powinien być pomijany (SKIP)"
print("\nOK: brak chunków typu `picture` (oczekiwane — SKIP).")

Liczba chunków: 521
  text         369
  table        108
  infographic  44

OK: brak chunków typu `picture` (oczekiwane — SKIP).


## 4. Rozkład wielkości chunków

In [4]:
def percentile(values: list[int], p: float) -> float:
    if not values:
        return 0.0
    s = sorted(values)
    k = (len(s) - 1) * p
    lo, hi = int(k), min(int(k) + 1, len(s) - 1)
    return s[lo] + (s[hi] - s[lo]) * (k - lo)


rows = []
for t in ("text", "table", "infographic"):
    lens = [len(c["search_text"]) for c in chunks if c["element_type"] == t]
    if not lens:
        continue
    rows.append(
        (
            t,
            len(lens),
            min(lens),
            int(percentile(lens, 0.25)),
            int(median(lens)),
            int(percentile(lens, 0.75)),
            max(lens),
            int(mean(lens)),
        )
    )

header = ("type", "n", "min", "p25", "p50", "p75", "max", "mean")
print("  ".join(f"{h:>12}" for h in header))
for r in rows:
    print("  ".join(f"{str(v):>12}" for v in r))

over_limit = [
    c for c in chunks if c["element_type"] == "text" and len(c["content"]) > MAX_CHARS
]
print(
    f"\nChunki tekstowe powyżej MAX_CHARS ({MAX_CHARS}): {len(over_limit)} "
    f"(oczekiwane 0 — split zawsze na granicy bloku)."
)

        type             n           min           p25           p50           p75           max          mean
        text           369           102          1195          1372          1478          2204          1279
       table           108           194           627           896          1211          3812          1127
 infographic            44           198           323           414           529          2754           574

Chunki tekstowe powyżej MAX_CHARS (1500): 3 (oczekiwane 0 — split zawsze na granicy bloku).


## 5. Sanity check: footnote / identifier w chunkach tekstowych

W v1 `footnote` był pomijany — w v2 ma się pojawiać w `content` chunka tekstowego.

In [5]:
footnote_bids = {b.block_id for b in doc.get_all_blocks() if b.element_type == "footnote"}
identifier_bids = {b.block_id for b in doc.get_all_blocks() if b.element_type == "identifier"}

text_chunks = [c for c in chunks if c["element_type"] == "text"]
chunked_sources = {bid for c in text_chunks for bid in c["source_blocks"]}

print(
    f"footnote bloków w dokumencie: {len(footnote_bids)} — "
    f"zapakowanych w chunki tekstowe: {len(footnote_bids & chunked_sources)}"
)
print(
    f"identifier bloków w dokumencie: {len(identifier_bids)} — "
    f"zapakowanych w chunki tekstowe: {len(identifier_bids & chunked_sources)}"
)

footnote bloków w dokumencie: 59 — zapakowanych w chunki tekstowe: 59
identifier bloków w dokumencie: 133 — zapakowanych w chunki tekstowe: 133


## 6. Podgląd mediów (caption jako prefiks treści)

In [6]:
MEDIA_PREVIEW = 5


def show(c: dict) -> None:
    print("=" * 80)
    print(
        f"[{c['chunk_index']}] type={c['element_type']}  page={c['page']}  "
        f"pages={c['pages']}  section={c['section']!r}  len={len(c['search_text'])}"
    )
    print("-" * 80)
    print(c["search_text"])
    print()


for t in ("table", "infographic"):
    print(f"\n### {t} (pierwsze {MEDIA_PREVIEW}):\n")
    for c in [x for x in chunks if x["element_type"] == t][:MEDIA_PREVIEW]:
        show(c)


### table (pierwsze 5):

[18] type=table  page=11  pages=[11]  section='6. Kluczowe dane finansowe'  len=730
--------------------------------------------------------------------------------
I Jedyny taki bank > 6. Kluczowe dane finansowe

| Tabela 1. Podstawowe parametry finansowe działalności Grupy Kapitałowej BGK (w mln zł)

Podstawowe parametry finansowe działalności Grupy Kapitałowej BGK w latach 2020–2024.
Wartości wyrażone w mln zł.

| dochodowość | 2024 | 2023 | 2022 | 2021 | 2020 |
|---|---|---|---|---|---|
| Wynik na działalności bankowej | 5 319 | 5 265 | 3 704 | 1 565 | 1 459 |
| Koszty administracyjne | -979 | -868 | -685 | -652 | -624 |
| Wynik z tytułu rezerw/odpisów | -673 | -399 | -301 | -332 | -385 |
| Udział w zyskach i stratach jednostek stowarzyszonych | 733 | 705 | 32 | 253 | -25 |
| Wynik brutto | 4 361 | 4 561 | 2 673 | 1 094 | 447 |
| Wynik netto | 3 562 | 3 732 | 2 162 | 875 | 367 |

[20] type=table  page=12  pages=[12]  section='6. Kluczowe dane finansowe'  l

## 7. Filtr po typie + długości

In [7]:
FILTER_TYPE = "text"  # "text", "table", "infographic"
FILTER_MIN_LEN = 0
FILTER_MAX_LEN = 10_000_000

filtered = [
    c
    for c in chunks
    if c["element_type"] == FILTER_TYPE
    and FILTER_MIN_LEN <= len(c["search_text"]) <= FILTER_MAX_LEN
]
print(
    f"Chunków typu {FILTER_TYPE!r} o długości [{FILTER_MIN_LEN}, {FILTER_MAX_LEN}]: {len(filtered)}\n"
)
for c in filtered:
    show(c)

Chunków typu 'text' o długości [0, 10000000]: 369

[0] type=text  page=4  pages=[4]  section='1. List Prezesa Zarządu'  len=1279
--------------------------------------------------------------------------------
I Jedyny taki bank > 1. List Prezesa Zarządu

GRI 2-22

Szanowni Państwo,

W minionym roku obchodziliśmy jubileusz stulecia BGK. To wielki przywilej kierować instytucją z tak wielkim dziedzictwem i znaczeniem dla rozwoju społeczno-gospodarczego kraju. Warto podkreślić jednak, że choć misja polskiego banku rozwoju pozostaje niezmieniona, w zależności od uwarunkowań gospodarczych podejmujemy działania, które w największy sposób odpowiadają na aktualne potrzeby.

Rok 2024 przyniósł wzrost PKB Polski o 2,9% r/r. Nasza gospodarka radzi sobie dobrze, zwłaszcza na tle innych państw UE. Jesteśmy jednak świadomi, że przed Europą stoją poważne wyzwania - utrzymanie konkurencyjności i rozwój innowacji. Sytuacja geopolityczna nie pozostawia złudzeń, że priorytetem Polski musi być wzmacnianie

## 8. Weryfikacja overlapu

Dla każdego chunka rozpoczynającego się od syntetycznego bloku `overlap-*`
sprawdzamy, że jego tekst jest faktycznym ogonem poprzedniego chunka (po
`_tail_text_for_overlap`).

In [8]:
from backend.app.document.vision_chunker import _tail_text_for_overlap

overlap_pairs = []
prev = None
for c in chunks:
    if (
        prev is not None
        and c["element_type"] == "text"
        and c["source_blocks"]
        and c["source_blocks"][0].startswith("overlap-")
        and prev["element_type"] == "text"
    ):
        expected = _tail_text_for_overlap(prev["content"], OVERLAP_CHARS)
        actual_head = c["content"].split("\n\n", 1)[0]
        overlap_pairs.append(
            {
                "prev_idx": prev["chunk_index"],
                "next_idx": c["chunk_index"],
                "match": expected == actual_head,
                "expected_len": len(expected),
                "actual_len": len(actual_head),
            }
        )
    prev = c

print(f"Par (prev → overlap): {len(overlap_pairs)}")
bad = [p for p in overlap_pairs if not p["match"]]
print(f"Niepasujące: {len(bad)} (oczekiwane 0)")
for p in overlap_pairs[:5]:
    print(p)

Par (prev → overlap): 250
Niepasujące: 86 (oczekiwane 0)
{'prev_idx': 0, 'next_idx': 1, 'match': True, 'expected_len': 193, 'actual_len': 193}
{'prev_idx': 1, 'next_idx': 2, 'match': True, 'expected_len': 134, 'actual_len': 134}
{'prev_idx': 2, 'next_idx': 3, 'match': True, 'expected_len': 184, 'actual_len': 184}
{'prev_idx': 3, 'next_idx': 4, 'match': True, 'expected_len': 113, 'actual_len': 113}
{'prev_idx': 4, 'next_idx': 5, 'match': True, 'expected_len': 118, 'actual_len': 118}


In [13]:
[c for c in chunks]

[{'search_text': 'I Jedyny taki bank > 1. List Prezesa Zarządu\n\nGRI 2-22\n\nSzanowni Państwo,\n\nW minionym roku obchodziliśmy jubileusz stulecia BGK. To wielki przywilej kierować instytucją z tak wielkim dziedzictwem i znaczeniem dla rozwoju społeczno-gospodarczego kraju. Warto podkreślić jednak, że choć misja polskiego banku rozwoju pozostaje niezmieniona, w zależności od uwarunkowań gospodarczych podejmujemy działania, które w największy sposób odpowiadają na aktualne potrzeby.\n\nRok 2024 przyniósł wzrost PKB Polski o 2,9% r/r. Nasza gospodarka radzi sobie dobrze, zwłaszcza na tle innych państw UE. Jesteśmy jednak świadomi, że przed Europą stoją poważne wyzwania - utrzymanie konkurencyjności i rozwój innowacji. Sytuacja geopolityczna nie pozostawia złudzeń, że priorytetem Polski musi być wzmacnianie odporności na zmiany i bezpieczeństwo strategiczne. Rozumieje je zarówno w kontekście militarnym, jak i inwestycji w sektorze energetycznym.\n\nW minionym roku wygenerowaliśmy 389 mld